In [3]:
import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact, FloatSlider, IntSlider
from IPython.display import display, clear_output

# --- Delta Robot Parameters (default) ---
base_radius = 200.0     # mm
end_radius = 50.0       # mm
upper_arm = 300.0       # mm
lower_arm = 600.0       # mm

base_joints = lambda br: [np.array([br * np.cos(theta), br * np.sin(theta), 0])
                          for theta in [0, 2*np.pi/3, 4*np.pi/3]]
end_joints_rel = lambda er: [np.array([er * np.cos(theta), er * np.sin(theta), 0])
                             for theta in [0, 2*np.pi/3, 4*np.pi/3]]


In [5]:
def compute_elbows_and_ee(x, y, z, br, er, ua):
    bj = base_joints(br)
    ej_rel = end_joints_rel(er)
    ee_center = np.array([x, y, z])
    ee_joints_world = [ee_center + ej for ej in ej_rel]
    elbows = []
    for b, ejw in zip(bj, ee_joints_world):
        # direction from base joint to end-joint
        dirv = ejw - b
        norm = np.linalg.norm(dirv)
        if norm == 0:
            dirv = np.array([0,0,-1.0])
            norm = 1.0
        dirv = dirv / norm
        elbow = b + ua * dirv
        elbows.append(elbow)
    return bj, elbows, ee_joints_world

def build_plot(bj, elbows, ee_joints_world):
    fig = go.Figure()
    # base joints
    for p in bj:
        fig.add_trace(go.Scatter3d(x=[p[0]], y=[p[1]], z=[p[2]], mode='markers', marker=dict(size=5, color='red')))
    # end-effector joints
    for p in ee_joints_world:
        fig.add_trace(go.Scatter3d(x=[p[0]], y=[p[1]], z=[p[2]], mode='markers', marker=dict(size=5, color='blue')))
    # arms
    for b, e, el in zip(bj, ee_joints_world, elbows):
        fig.add_trace(go.Scatter3d(x=[b[0], el[0]], y=[b[1], el[1]], z=[b[2], el[2]], mode='lines', line=dict(width=4)))
        fig.add_trace(go.Scatter3d(x=[el[0], e[0]], y=[el[1], e[1]], z=[el[2], e[2]], mode='lines', line=dict(width=4, dash='dash')))
    # platform outline
    xs = [p[0] for p in ee_joints_world] + [ee_joints_world[0][0]]
    ys = [p[1] for p in ee_joints_world] + [ee_joints_world[0][1]]
    zs = [p[2] for p in ee_joints_world] + [ee_joints_world[0][2]]
    fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode='lines', line=dict(color='blue', width=3)))
    fig.update_layout(scene=dict(aspectmode='data',
                                xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)'),
                      margin=dict(l=0,r=0,b=0,t=0), width=800, height=600)
    return fig


In [2]:
def show_delta(x=0.0, y=0.0, z=-400.0, br=200.0, er=50.0, ua=300.0):
    bj, elbows, ee = compute_elbows_and_ee(x, y, z, br, er, ua)
    fig = build_plot(bj, elbows, ee)
    fig.show()

# interactive sliders
interact(show_delta,
         x=FloatSlider(min=-250, max=250, step=5, value=0.0, description='X (mm)'),
         y=FloatSlider(min=-250, max=250, step=5, value=0.0, description='Y (mm)'),
         z=FloatSlider(min=-900, max=-200, step=5, value=-400.0, description='Z (mm)'),
         br=FloatSlider(min=80, max=400, step=1, value=200.0, description='Base R'),
         er=FloatSlider(min=10, max=120, step=1, value=50.0, description='EE R'),
         ua=FloatSlider(min=100, max=450, step=1, value=300.0, description='Upper arm'))


NameError: name 'interact' is not defined

In [8]:
# Replace the previous interactive cell with this one
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

out = widgets.Output(layout=widgets.Layout(width='820px', height='620px', overflow='auto'))

def update_and_show_html(x=0.0, y=0.0, z=-400.0, br=200.0, er=50.0, ua=300.0):
    with out:
        clear_output(wait=True)
        bj, elbows, ee = compute_elbows_and_ee(x, y, z, br, er, ua)
        fig = build_plot(bj, elbows, ee)
        # Convert the figure to an embeddable HTML fragment (Plotly JS loaded from CDN)
        html_fragment = fig.to_html(full_html=False, include_plotlyjs='cdn')
        display(HTML(html_fragment))

# Create sliders (continuous_update=True may be slower; set to False if you prefer)
sliders = {
    'x': widgets.FloatSlider(min=-250, max=250, step=5, value=0.0, description='X (mm)', continuous_update=False),
    'y': widgets.FloatSlider(min=-250, max=250, step=5, value=0.0, description='Y (mm)', continuous_update=False),
    'z': widgets.FloatSlider(min=-900, max=-200, step=5, value=-400.0, description='Z (mm)', continuous_update=False),
    'br': widgets.FloatSlider(min=80, max=400, step=1, value=200.0, description='Base R', continuous_update=False),
    'er': widgets.FloatSlider(min=10, max=120, step=1, value=50.0, description='EE R', continuous_update=False),
    'ua': widgets.FloatSlider(min=100, max=450, step=1, value=300.0, description='Upper arm', continuous_update=False),
}

controls = widgets.VBox([sliders['x'], sliders['y'], sliders['z'], sliders['br'], sliders['er'], sliders['ua']])

def on_change(change):
    # read current slider values and update
    vals = {k: sliders[k].value for k in sliders}
    update_and_show_html(vals['x'], vals['y'], vals['z'], vals['br'], vals['er'], vals['ua'])

for s in sliders.values():
    s.observe(on_change, names='value')

# initial render
on_change(None)

# layout: controls left, figure right
display(widgets.HBox([controls, out]))
